# 02. Feature Engineering

Objetivo:
 - Construir features dinámicas sin data leakage
 - Implementar Elo rating desde cero
 - Generar dataset listo para modelado


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

fullPath = Path("../data/processed/full_matches.csv")
outputPath = Path("../data/processed/model_dataset.csv")

df = pd.read_csv(fullPath, parse_dates=["date"])
df = df.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Dataset cargado:", df.shape)

Dataset cargado: (49069, 12)


## 1. Inicializar estructuras

In [14]:
teams = pd.Index(df["homeTeam"]).union(pd.Index(df["awayTeam"])).unique()

# Elo inicial
eloDict = {team: 1500 for team in teams}

# Stats acumuladas
stats = {
    team: {
        "games": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goalsFor": 0,
        "goalsAgainst": 0,
    }
    for team in teams
}

## 2. Funciones clave

In [15]:
def expectedScore(ratingA, ratingB):
    return 1 / (1 + 10 ** ((ratingB - ratingA) / 400))

def updateElo(rA, rB, scoreA, k):
    expectedA = expectedScore(rA, rB)
    return rA + k * (scoreA - expectedA)

def getKFactor(tournamentWeight):
    return 20 * tournamentWeight


## 3. Feature Engineering secuencial

In [16]:
rows = []

for _, row in df.iterrows():
    home = row["homeTeam"]
    away = row["awayTeam"]

    # Elo actual (ANTES)
    eloHome = eloDict[home]
    eloAway = eloDict[away]

    # Stats actuales
    h = stats[home]
    a = stats[away]

    # Juegos
    homeGames = h["games"]
    awayGames = a["games"]

    # Rates seguros
    def safe_div(num, den):
        return num / den if den > 0 else np.nan

    homeWinRate = safe_div(h["wins"], homeGames)
    awayWinRate = safe_div(a["wins"], awayGames)

    homeGoalsForAvg = safe_div(h["goalsFor"], homeGames)
    awayGoalsForAvg = safe_div(a["goalsFor"], awayGames)

    homeGoalsAgainstAvg = safe_div(h["goalsAgainst"], homeGames)
    awayGoalsAgainstAvg = safe_div(a["goalsAgainst"], awayGames)

    # Features principales
    eloDiff = eloHome - eloAway
    absEloDiff = abs(eloDiff)
    expectedHomeWin = expectedScore(eloHome, eloAway)

    attackDiff = homeGoalsForAvg - awayGoalsAgainstAvg
    defenseDiff = homeGoalsAgainstAvg - awayGoalsForAvg

    # Guardar ANTES de actualizar (NO LEAKAGE)
    rows.append({
        "date": row["date"],
        "homeTeam": home,
        "awayTeam": away,

        "eloHome": eloHome,
        "eloAway": eloAway,
        "eloDiff": eloDiff,
        "absEloDiff": absEloDiff,
        "eloExpectedHomeWin": expectedHomeWin,

        "homeMatchesPlayed": homeGames,
        "awayMatchesPlayed": awayGames,

        "homeWinRate": homeWinRate,
        "awayWinRate": awayWinRate,
        "winRateDiff": homeWinRate - awayWinRate,

        "homeGoalsForAvg": homeGoalsForAvg,
        "awayGoalsForAvg": awayGoalsForAvg,
        "goalsForDiff": homeGoalsForAvg - awayGoalsForAvg,

        "homeGoalsAgainstAvg": homeGoalsAgainstAvg,
        "awayGoalsAgainstAvg": awayGoalsAgainstAvg,
        "goalsAgainstDiff": homeGoalsAgainstAvg - awayGoalsAgainstAvg,

        "attackDiff": attackDiff,
        "defenseDiff": defenseDiff,

        "tournamentWeight": row["tournamentWeight"],
        "neutral": int(row["neutral"]),

        "target": row["matchResult"],
    })

    # --- UPDATE ---
    homeScore = row["homeScore"]
    awayScore = row["awayScore"]

    if homeScore > awayScore:
        sHome = 1
        h["wins"] += 1
        a["losses"] += 1
    elif homeScore < awayScore:
        sHome = 0
        h["losses"] += 1
        a["wins"] += 1
    else:
        sHome = 0.5
        h["draws"] += 1
        a["draws"] += 1

    k = getKFactor(row["tournamentWeight"])

    eloDict[home] = updateElo(eloHome, eloAway, sHome, k)
    eloDict[away] = updateElo(eloAway, eloHome, 1 - sHome, k)

    # actualizar stats
    h["games"] += 1
    a["games"] += 1

    h["goalsFor"] += homeScore
    h["goalsAgainst"] += awayScore

    a["goalsFor"] += awayScore
    a["goalsAgainst"] += homeScore

# %%
featureDf = pd.DataFrame(rows)



In [17]:
print("Feature dataset:", featureDf.shape)
featureDf.head()

Feature dataset: (49069, 24)


,date,homeTeam,awayTeam,eloHome,eloAway,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,homeWinRate,awayWinRate,winRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstDiff,attackDiff,defenseDiff,tournamentWeight,neutral,target
0,1872-11-30,Scotland,England,1500.000000,1500.000000,0.000000,0.000000,0.500000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,0,draw
1,1873-03-08,England,Scotland,1500.000000,1500.000000,0.000000,0.000000,0.500000,1,1,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.3,0,homeWin
2,1874-03-07,Scotland,England,1497.000000,1503.000000,-6.000000,6.000000,0.491366,2,2,0.000000,0.500000,-0.5,1.000000,2.000000,-1.000000,2.000000,1.000000,1.000000,0.0,0.0,0.3,0,homeWin
3,1875-03-06,England,Scotland,1499.948197,1500.051803,-0.103606,0.103606,0.499851,3,3,0.333333,0.333333,0.0,1.666667,1.333333,0.333333,1.333333,1.666667,-0.333333,0.0,0.0,0.3,0,draw
4,1876-03-04,Scotland,England,1500.050908,1499.949092,0.101817,0.101817,0.500147,4,4,0.250000,0.250000,0.0,1.500000,1.750000,-0.250000,1.750000,1.500000,0.250000,0.0,0.0,0.3,0,homeWin


## 4. Limpieza final

In [18]:
# 1. Filtrar equipos con historial suficiente
minGames = 5

modelDf = featureDf[
    (featureDf["homeMatchesPlayed"] >= minGames) &
    (featureDf["awayMatchesPlayed"] >= minGames)
].copy()

print("Tras filtro de experiencia:", modelDf.shape)

Tras filtro de experiencia: (47769, 24)


In [19]:
# 2. Eliminar NaNs
modelDf = modelDf.dropna().reset_index(drop=True)

print("Tras eliminar NaNs:", modelDf.shape)

Tras eliminar NaNs: (47769, 24)


In [20]:
# 3. Verificación final
print("NaNs restantes:", modelDf.isnull().sum().sum())

NaNs restantes: 0


## 5. Exportar dataset

In [21]:
outputPath.parent.mkdir(parents=True, exist_ok=True)
modelDf.to_csv(outputPath, index=False)

print("Dataset final guardado en:", outputPath)
print("Shape final:", modelDf.shape)

modelDf.head()

Dataset final guardado en: ..\data\processed\model_dataset.csv
Shape final: (47769, 24)


,date,homeTeam,awayTeam,eloHome,eloAway,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,homeWinRate,awayWinRate,winRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstDiff,attackDiff,defenseDiff,tournamentWeight,neutral,target
0,1877-03-03,England,Scotland,1496.949971,1506.023694,-9.073723,9.073723,0.486945,5,6,0.200000,0.500000,-0.300000,1.400000,2.166667,-0.766667,1.80,1.166667,0.633333,0.233333,-0.366667,0.3,0,awayWin
1,1878-03-02,Scotland,England,1511.842486,1494.028302,17.814184,17.814184,0.525614,8,6,0.625000,0.166667,0.458333,2.250000,1.333333,0.916667,1.00,2.000000,-1.000000,0.250000,-0.333333,0.3,0,homeWin
2,1879-04-05,England,Scotland,1494.183063,1517.511482,-23.328419,23.328419,0.466478,8,10,0.250000,0.700000,-0.450000,1.500000,3.400000,-1.900000,2.50,1.000000,1.500000,0.500000,-0.900000,0.3,0,homeWin
3,1880-03-13,Scotland,England,1517.086224,1497.384194,19.702030,19.702030,0.528323,12,9,0.666667,0.333333,0.333333,3.416667,1.888889,1.527778,1.25,2.666667,-1.416667,0.750000,-0.638889,0.3,0,homeWin
4,1880-03-15,Wales,England,1485.529582,1494.554133,-9.024551,9.024551,0.487016,5,10,0.000000,0.300000,-0.300000,0.200000,2.100000,-1.900000,4.00,2.900000,1.100000,-2.700000,1.900000,0.3,0,awayWin
